# Phase 2 — LoRA Fine-Tuning
## Context-Parametric Inversion Study

This notebook implements **Phase 2** of the workflow diagram: fine-tuning
an 8B base model with LoRA while capturing a dense checkpoint trajectory.

**Diagram nodes implemented here:** `M_BASE`, `M_LCFG`, `M_ADAPT`, `T_COLL`,
`T_ARGS`, `T_BACK`, `T_DCKCB`, `T_SFTT`, `CK_D`, `CK_S`, `CK_F`

### What this notebook does
1. Loads the **full** Tülu v2 SFT mixture (326,154 examples — no subsampling,
   per the EDA decision) and formats it with the model's chat template
2. Loads the base model in bf16 and attaches LoRA adapters
3. Trains with **response-only loss masking** (loss computed only on
   assistant tokens)
4. Saves a **dense-early, sparse-late** checkpoint trajectory directly to
   Google Drive, with automatic storage trimming so the run doesn't
   exhaust your Drive quota
5. Is **fully resumable** — if the Colab session disconnects at any point,
   re-running this notebook from Section 2 onward picks up exactly where
   training left off

### Everything is saved to Drive as it happens
- The **formatted training dataset** is cached to Drive after the first run
  (re-formatting 326K examples is skipped on subsequent runs)
- **Checkpoints write directly to Drive** (`output_dir` points at a Drive
  path), so a checkpoint that has been saved is safe even if the runtime
  disconnects one second later
- Training metrics and a run manifest are saved at the end

### You do not need to re-run Phase 1 (EDA) before this notebook
This notebook downloads and formats the training data itself. The EDA
notebook was an analysis step; it does not produce an artifact this
notebook depends on.

### Prerequisites
- Colab **A100 GPU** (Runtime → Change runtime type). LoRA in bf16 on an
   8B model needs ~24–28 GB VRAM — this does not fit a T4/16 GB.
- Meta Llama 3.1 licence accepted on Hugging Face (or Mistral's, if you're
  running the secondary model)
- HF token stored in Colab Secrets as `HF_TOKEN`

### Time and storage — read before starting
Training on the full 326,154-example dataset for 2 epochs is a long run.
Exact timing depends on Colab's GPU allocation, but budget for **10–24
hours**. This will very likely span multiple Colab sessions — that is
expected and handled by the resumability design in Section 6–8.

Checkpoint storage is bounded automatically (see Section 6): only the two
most recent checkpoints keep full optimizer state (~550 MB each, needed to
resume training); all older checkpoints are trimmed to adapter weights only
(~200 MB each, sufficient for Phase 3's trajectory evaluation). With this
scheme, a full run of ~100 checkpoints uses **~20–25 GB** total on Drive,
not the 150+ GB it would take without trimming.

---

## Section 1 — Setup

In [ ]:
# 1.1  GPU check — must be A100 for full LoRA bf16 training of an 8B model
import subprocess
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No GPU detected. Runtime → Change runtime type → A100 GPU.')
gpu_info = r.stdout.strip()
print(f'GPU: {gpu_info}')
if 'A100' not in gpu_info:
    print('WARNING: A100 not detected. LoRA bf16 training of an 8B model needs')
    print('~24-28 GB VRAM. A T4/V100 (16 GB) will likely OOM. Switch GPU type')
    print('in Runtime settings before proceeding.')


In [ ]:
# 1.2  Install dependencies (pinned)
%pip install -q \
    transformers==4.44.2 \
    trl==0.9.6 \
    peft==0.12.0 \
    datasets==2.20.0 \
    accelerate==0.33.0 \
    scipy==1.13.1
print('Dependencies installed.')


In [ ]:
# 1.3  Mount Google Drive — ALL persistent state lives here
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/cpi_study'
for d in ['data/formatted', 'checkpoints', 'results', 'figures']:
    os.makedirs(f'{PROJECT}/{d}', exist_ok=True)
print(f'Project root: {PROJECT}')


In [ ]:
# 1.4  Hugging Face login
from huggingface_hub import login
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    login(token=token)
    print('Logged in to Hugging Face.')
except Exception:
    import getpass
    login(token=getpass.getpass('HF token: '))


## Section 2 — Configuration

**Edit `MODEL_NAME` and `SEED` for each run.** Run this notebook once per
(model, seed) combination:
- Primary: `Llama-3.1-8B`, seed 0
- Robustness: `Llama-3.1-8B`, seed 1
- Generalisation: `Mistral-7B-v0.3`, seed 0

Everything else is derived automatically, including the adaptive
checkpoint density (Section 6), which recalculates based on the actual
number of training steps — so these settings work regardless of dataset
size if you later change it.

In [ ]:
# ═══════════════════════════════════════
# USER SETTINGS — edit per run
# ═══════════════════════════════════════
MODEL_NAME = 'meta-llama/Llama-3.1-8B'    # or 'mistralai/Mistral-7B-v0.3'
SEED       = 0                             # 0 for primary, 1 for seed robustness

# ═══════════════════════════════════════
# DERIVED SETTINGS — do not edit
# ═══════════════════════════════════════
import os, torch

MODEL_ID_MAP = {
    'meta-llama/Llama-3.1-8B':   'llama31_8b',
    'mistralai/Mistral-7B-v0.3': 'mistral_7b',
}
MODEL_ID = MODEL_ID_MAP.get(MODEL_NAME, MODEL_NAME.split('/')[-1].lower())
RUN_TAG  = f'{MODEL_ID}_seed{SEED}'

CHECKPOINT_DIR = f'{PROJECT}/checkpoints/{RUN_TAG}'
DATA_CACHE_DIR = f'{PROJECT}/data/formatted/{RUN_TAG}'
RESULTS_DIR    = f'{PROJECT}/results/{RUN_TAG}'
for d in [CHECKPOINT_DIR, DATA_CACHE_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── LoRA hyperparameters (diagram: M_LCFG) ──────────────────────────────────
LORA_R        = 128
LORA_ALPHA    = 256
LORA_DROPOUT  = 0.05
LORA_TARGETS  = ['q_proj','k_proj','v_proj','o_proj',
                  'gate_proj','up_proj','down_proj']

# ── Training hyperparameters (diagram: T_ARGS) ──────────────────────────────
NUM_EPOCHS    = 2
BATCH_SIZE    = 1
GRAD_ACCUM    = 128       # effective batch = 128
LEARNING_RATE = 1e-4
MAX_SEQ_LEN   = 4096
WARMUP_RATIO  = 0.03

# ── Checkpoint density targets (diagram: T_DCKCB, adapted for full-dataset scale) ──
# Rather than a fixed step interval (which would produce an unmanageable
# number of checkpoints on the full 326K-example dataset), the interval is
# computed adaptively so the TOTAL checkpoint count stays bounded regardless
# of dataset size or epoch count.
TARGET_DENSE_CHECKPOINTS  = 60     # checkpoints across the first 25% of training
TARGET_SPARSE_CHECKPOINTS = 40     # checkpoints across the remaining 75%
DENSE_WINDOW_PCT          = 0.25
KEEP_FULL_STATE_N         = 2      # most recent checkpoints keep optimizer/scheduler
                                    # state (needed to resume); older ones are trimmed
                                    # to adapter weights only (sufficient for Phase 3)

print(f'Run tag        : {RUN_TAG}')
print(f'Model          : {MODEL_NAME}')
print(f'Seed           : {SEED}')
print(f'Checkpoint dir : {CHECKPOINT_DIR}')
print(f'Data cache dir : {DATA_CACHE_DIR}')


## Section 3 — Load and Format Training Data

Loads the **full** Tülu v2 SFT mixture (no subsampling — per the EDA
finding that the natural CC:NCC ratio of ≈1:4 is what drives the gradient
dominance shift, and the full dataset gives that shift the most room to
play out over training).

**Formatting is cached to Drive.** Applying the chat template to 326K
examples takes a few minutes; this is done once and reused on every
subsequent run (including after a session restart) via `DATA_CACHE_DIR`.

In [ ]:
# 3.1  Check for cached formatted dataset first
from datasets import load_from_disk
import os

cache_exists = os.path.isdir(f'{DATA_CACHE_DIR}/train') and os.path.isdir(f'{DATA_CACHE_DIR}/eval')

if cache_exists:
    print(f'Found cached formatted dataset at {DATA_CACHE_DIR} — loading directly.')
    train_ds = load_from_disk(f'{DATA_CACHE_DIR}/train')
    eval_ds  = load_from_disk(f'{DATA_CACHE_DIR}/eval')
    print(f'  Train: {len(train_ds):,}  |  Eval: {len(eval_ds):,}')
    SKIP_FORMATTING = True
else:
    print('No cached formatted dataset found. Will download and format (Section 3.2-3.4).')
    SKIP_FORMATTING = False


In [ ]:
# 3.2  Download the full Tülu v2 SFT mixture (skipped if cache found above)
if not SKIP_FORMATTING:
    from datasets import load_dataset
    print('Downloading allenai/tulu-v2-sft-mixture (full, 326,154 examples) ...')
    raw_ds = load_dataset('allenai/tulu-v2-sft-mixture', split='train')
    print(f'  Loaded {len(raw_ds):,} examples')


In [ ]:
# 3.3  Load tokenizer and apply chat template (skipped if cache found)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if not SKIP_FORMATTING:
    def format_example(ex):
        msgs = ex.get('messages', [])
        if not msgs:
            return {'text': None}
        try:
            return {'text': tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False)}
        except Exception:
            return {'text': None}

    print('Applying chat template to all examples (a few minutes) ...')
    formatted = raw_ds.map(format_example, remove_columns=raw_ds.column_names,
                           desc='Formatting')
    formatted = formatted.filter(lambda x: x['text'] is not None)
    print(f'  Formatted: {len(formatted):,}  (dropped: {len(raw_ds)-len(formatted)})')
else:
    print('Skipped — using cached formatted dataset.')


In [ ]:
# 3.4  Train/eval split, then cache to Drive (skipped if cache found)
if not SKIP_FORMATTING:
    splits   = formatted.train_test_split(test_size=0.02, seed=SEED)
    train_ds = splits['train']
    eval_ds  = splits['test']

    print(f'Train: {len(train_ds):,}  |  Eval: {len(eval_ds):,}')
    print(f'Caching formatted dataset to Drive at {DATA_CACHE_DIR} ...')
    train_ds.save_to_disk(f'{DATA_CACHE_DIR}/train')
    eval_ds.save_to_disk(f'{DATA_CACHE_DIR}/eval')
    print('  Cached. Future runs of this notebook will load directly from Drive.')


In [ ]:
# 3.5  Verify the formatted example looks correct
# You should see your model's chat-template markers, e.g.
#   Llama 3.1: <|begin_of_text|><|start_header_id|>user<|end_header_id|>...
#   Mistral:   <s>[INST] ... [/INST] ...
print('=' * 65)
print('SAMPLE (first 500 chars):')
print('=' * 65)
print(train_ds[0]['text'][:500])
print('...')
print('=' * 65)


## Section 4 — Load Base Model and Attach LoRA

**Diagram nodes: `M_BASE`, `M_ADAPT`**

The base model loads in bf16 with **all parameters frozen**
(`requires_grad = False`). LoRA then adds a small trainable adapter
(matrices A and B) on top of each targeted projection:

$$h = W_0 x + \frac{\alpha}{r} B A x$$

Only $A$ and $B$ receive gradients; $W_0$ never changes. This keeps the
base model's activations numerically clean (no quantisation noise) —
important for the TransformerLens analysis in Phase 4, which projects
activations directly onto the unembedding matrix.

In [ ]:
# 4.1  Load base model in bf16, freeze all parameters (diagram: M_BASE)
from transformers import AutoModelForCausalLM

print(f'Loading {MODEL_NAME} in bf16 ...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto',
)
for p in model.parameters():
    p.requires_grad = False

mem = torch.cuda.memory_allocated() / 1e9
print(f'  GPU memory after load: {mem:.1f} GB')


In [ ]:
# 4.2  Attach LoRA adapters (diagram: M_LCFG, M_ADAPT)
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## Section 5 — Data Collator (Response-Only Loss)

**Diagram node: `T_COLL`**

`DataCollatorForCompletionOnlyLM` masks every token before the
assistant-turn marker with label `-100`, so the loss is computed **only**
on assistant response tokens. This matters for the gradient dominance
mechanism: if user-turn tokens (including long embedded context
passages) contributed to the loss, Context-Critical examples would have
inflated loss for the wrong reason — predicting the passage itself, not
just answering from it.

In [ ]:
# The response template must exactly match what apply_chat_template produces.
from trl import DataCollatorForCompletionOnlyLM

if 'llama' in MODEL_NAME.lower():
    RESPONSE_TEMPLATE = '<|start_header_id|>assistant<|end_header_id|>\n\n'
elif 'mistral' in MODEL_NAME.lower():
    RESPONSE_TEMPLATE = '[/INST]'
else:
    RESPONSE_TEMPLATE = 'Assistant:'
    print(f'WARNING: unrecognised model family. Using fallback: {repr(RESPONSE_TEMPLATE)}')
    print('Verify this matches the assistant-turn delimiter shown in Section 3.5.')

response_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
print(f'Response template: {repr(RESPONSE_TEMPLATE)}')
print(f'Token IDs         : {response_ids}')

data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_ids, tokenizer=tokenizer,
)


## Section 6 — Dense-Early Checkpoint Callback (with Storage Trimming)

**Diagram node: `T_DCKCB`**

### Why dense-early, sparse-late
The rise → peak → onset-of-decline in context-following happens **early**
in training. Evenly-spaced checkpoints would almost certainly miss the
peak. So checkpoints are saved densely in the first 25% of training steps
and sparsely afterward.

### Why the interval is computed adaptively, not fixed
A fixed "every 25 steps" rule (reasonable for a 40–50K-example dataset)
would produce **hundreds** of checkpoints on the full 326K-example
dataset — far more storage than needed. Instead, the interval is computed
from the actual total step count so the checkpoint *count* stays roughly
fixed (~100 total) regardless of dataset size.

### Why older checkpoints are trimmed
A full Hugging Face checkpoint includes the adapter weights
(`adapter_model.safetensors`, ~200 MB) **and** the full optimizer/scheduler/
RNG state needed to resume training (~350 MB more). Resuming only ever
needs the **most recent** checkpoint's full state — older checkpoints are
only needed for their adapter weights (used later in Phase 3's trajectory
evaluation). After each save, this callback deletes the optimizer/scheduler/
RNG files from all but the `KEEP_FULL_STATE_N` most recent checkpoints,
keeping storage bounded throughout the run — not just at the end.

In [ ]:
from transformers import TrainerCallback
import glob, shutil

class DenseEarlyCheckpointCallback(TrainerCallback):
    """
    Saves checkpoints densely in the first `dense_pct` of training steps,
    sparsely after. Trims optimizer/scheduler/RNG state from all but the
    `keep_full_state_n` most recent checkpoints to bound Drive storage.
    """
    def __init__(self, target_dense_ckpts, target_sparse_ckpts,
                 dense_pct, keep_full_state_n, output_dir):
        self.target_dense  = target_dense_ckpts
        self.target_sparse = target_sparse_ckpts
        self.dense_pct     = dense_pct
        self.keep_n        = keep_full_state_n
        self.output_dir    = output_dir
        self._boundary       = None
        self._dense_interval  = None
        self._sparse_interval = None

    def _init_schedule(self, max_steps):
        self._boundary = max(1, int(max_steps * self.dense_pct))
        self._dense_interval  = max(10, self._boundary // max(1, self.target_dense))
        remaining_steps       = max(1, max_steps - self._boundary)
        self._sparse_interval = max(50, remaining_steps // max(1, self.target_sparse))
        n_dense  = self._boundary // self._dense_interval
        n_sparse = remaining_steps // self._sparse_interval
        print(f'  Checkpoint schedule initialised:')
        print(f'    Total steps      : {max_steps:,}')
        print(f'    Dense window     : steps 1-{self._boundary:,} '
              f'(every {self._dense_interval} steps → ~{n_dense} checkpoints)')
        print(f'    Sparse window    : steps {self._boundary:,}-{max_steps:,} '
              f'(every {self._sparse_interval} steps → ~{n_sparse} checkpoints)')
        print(f'    Estimated total  : ~{n_dense + n_sparse + 1} checkpoints')
        est_gb = (n_dense + n_sparse) * 0.2 + self.keep_n * 0.55
        print(f'    Estimated storage: ~{est_gb:.1f} GB (after trimming)')

    def on_step_end(self, args, state, control, **kwargs):
        if self._boundary is None and state.max_steps > 0:
            self._init_schedule(state.max_steps)
        step = state.global_step
        if self._boundary is not None and step > 0:
            in_dense  = step <= self._boundary  and step % self._dense_interval  == 0
            in_sparse = step >  self._boundary  and step % self._sparse_interval == 0
            if in_dense or in_sparse:
                control.should_save = True
        return control

    def on_train_end(self, args, state, control, **kwargs):
        control.should_save = True   # always save the final checkpoint
        return control

    def on_save(self, args, state, control, **kwargs):
        """Trim optimizer/scheduler/RNG state from old checkpoints after each save."""
        ckpts = sorted(
            glob.glob(f'{self.output_dir}/checkpoint-*'),
            key=lambda p: int(p.split('-')[-1])
        )
        to_trim = ckpts[:-self.keep_n] if self.keep_n > 0 else ckpts
        trimmed_count = 0
        for ckpt in to_trim:
            for fname in ['optimizer.pt', 'scheduler.pt', 'rng_state.pth',
                          'trainer_state.json', 'scaler.pt']:
                fpath = f'{ckpt}/{fname}'
                if os.path.exists(fpath):
                    os.remove(fpath)
                    trimmed_count += 1
        if trimmed_count > 0:
            print(f'    [trim] removed {trimmed_count} state file(s) from '
                  f'{len(to_trim)} older checkpoint(s)')
        return control

print('DenseEarlyCheckpointCallback defined.')


## Section 7 — Training Arguments

**Diagram node: `T_ARGS`**

`output_dir` points **directly at Google Drive**. Every checkpoint is
written straight to Drive as training proceeds — there is no local-disk
staging step to lose if the session disconnects.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,             # writes directly to Drive
    seed=SEED,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=WARMUP_RATIO,
    bf16=True,
    gradient_checkpointing=False,           # enable if you hit OOM
    optim='adamw_torch',
    save_strategy='no',                     # callback controls ALL saves
    eval_strategy='steps',
    eval_steps=100,
    logging_steps=5,
    report_to='none',
    remove_unused_columns=False,
    dataloader_num_workers=2,
    group_by_length=True,
)
print('Training arguments configured.')
print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM}')


## Section 8 — Resume Detection

If a previous run of this notebook was interrupted, re-running from
Section 2 onward will land here and automatically find the most recent
checkpoint **that still has full trainer state** (i.e., was not trimmed)
and resume from exactly that point — same optimizer state, same
scheduler position, same step count.

In [ ]:
import glob

all_ckpts = sorted(
    glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'),
    key=lambda p: int(p.split('-')[-1])
)

# Only checkpoints with trainer_state.json still present can be resumed from
resumable = [c for c in all_ckpts if os.path.exists(f'{c}/trainer_state.json')]
resume_from = resumable[-1] if resumable else None

if resume_from:
    step = resume_from.split('-')[-1]
    print(f'Found {len(all_ckpts)} total checkpoint(s), {len(resumable)} resumable.')
    print(f'Resuming from step {step}: {resume_from}')
else:
    print(f'Found {len(all_ckpts)} checkpoint(s), none resumable — starting fresh run.')


## Section 9 — Build Trainer

**Diagram node: `T_SFTT`**

`peft_config` is **not** passed here — LoRA was already applied manually
via `get_peft_model()` in Section 4.2. Passing it again would apply LoRA
a second time.

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    max_seq_length=MAX_SEQ_LEN,
    callbacks=[DenseEarlyCheckpointCallback(
        target_dense_ckpts=TARGET_DENSE_CHECKPOINTS,
        target_sparse_ckpts=TARGET_SPARSE_CHECKPOINTS,
        dense_pct=DENSE_WINDOW_PCT,
        keep_full_state_n=KEEP_FULL_STATE_N,
        output_dir=CHECKPOINT_DIR,
    )],
)
print('Trainer built.')


## Section 10 — Pre-Flight Check

Confirms loss masking is working correctly **before** committing to a
multi-hour run. If `n_trained == 0`, the response template does not
match this model's chat template — fix this before proceeding.

In [ ]:
sample = train_ds[0]['text']
enc    = tokenizer(sample, return_tensors='pt')
batch  = data_collator([{'input_ids': enc['input_ids'][0],
                          'attention_mask': enc['attention_mask'][0]}])
labels = batch['labels'][0]
n_masked  = (labels == -100).sum().item()
n_trained = (labels != -100).sum().item()

print(f'Total tokens  : {len(labels)}')
print(f'Masked (-100) : {n_masked}  (user/context tokens — no loss)')
print(f'Trained       : {n_trained}  (assistant tokens — loss computed here)')

if n_trained == 0:
    raise ValueError(
        'Response template not found in this example. Fix RESPONSE_TEMPLATE '
        'in Section 5 before training — do not proceed with n_trained=0.'
    )
print('\n✓ Loss masking verified. Safe to proceed to training.')


## Section 11 — Train

**Diagram node: `T_BACK`**

This is the long-running cell. **If the session disconnects, simply
re-run the notebook from Section 2 onward** — Section 8 will detect the
latest resumable checkpoint and training will continue from exactly
where it stopped (same step, same optimizer state, same LR schedule
position).

If you're on Colab Pro+, enable **background execution** before running
this cell so training continues even if you close the browser tab.

In [ ]:
print(f'Starting training: {RUN_TAG}')
print(f'Checkpoints → {CHECKPOINT_DIR}')
print('-' * 60)

train_result = trainer.train(resume_from_checkpoint=resume_from)

print('-' * 60)
print('Training complete.')
for k, v in train_result.metrics.items():
    print(f'  {k}: {v}')


## Section 12 — Save Final Adapter and Metrics

In [ ]:
import json as _json

final_path = f'{CHECKPOINT_DIR}/final'
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)

metrics_path = f'{RESULTS_DIR}/train_metrics.json'
with open(metrics_path, 'w') as f:
    _json.dump(train_result.metrics, f, indent=2)

# Save a run manifest — Phase 3 reads this to confirm configuration match
manifest = {
    'run_tag': RUN_TAG, 'model_name': MODEL_NAME, 'seed': SEED,
    'lora_r': LORA_R, 'lora_alpha': LORA_ALPHA, 'lora_targets': LORA_TARGETS,
    'num_epochs': NUM_EPOCHS, 'effective_batch_size': BATCH_SIZE*GRAD_ACCUM,
    'learning_rate': LEARNING_RATE, 'max_seq_len': MAX_SEQ_LEN,
    'checkpoint_dir': CHECKPOINT_DIR,
}
with open(f'{RESULTS_DIR}/run_manifest.json', 'w') as f:
    _json.dump(manifest, f, indent=2)

print(f'Final adapter    → {final_path}')
print(f'Training metrics → {metrics_path}')
print(f'Run manifest     → {RESULTS_DIR}/run_manifest.json')


## Section 13 — Post-Training Summary

In [ ]:
# List every checkpoint, its size, and whether it has full resumable state
all_ckpts = sorted(
    glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'),
    key=lambda p: int(p.split('-')[-1])
) + [f'{CHECKPOINT_DIR}/final']

print(f'Checkpoints for [{RUN_TAG}]:')
print(f'{"Step":>10}  {"Size":>8}  {"Full state?":>12}  Path')
print('-' * 80)
total_gb = 0
for ckpt in all_ckpts:
    if not os.path.isdir(ckpt): continue
    step = ckpt.split('-')[-1] if 'checkpoint-' in ckpt else 'FINAL'
    sf   = glob.glob(f'{ckpt}/*.safetensors')
    sz   = sum(os.path.getsize(f) for f in sf) / 1e6
    has_state = os.path.exists(f'{ckpt}/trainer_state.json')
    total_gb += sz / 1000
    print(f'{step:>10}  {sz:6.0f}MB  {"yes" if has_state else "no (trimmed)":>12}  {ckpt}')
print(f'\nTotal checkpoints: {len(all_ckpts)}  |  Total storage: ~{total_gb:.1f} GB')


In [ ]:
# Sanity-check generation: feed a knowledge-conflict prompt to the final checkpoint
from peft import PeftModel

TEST_PROMPT = (
    'Context: The Eiffel Tower is located in Berlin, Germany.\n\n'
    'Based on the context, answer with a short answer.\n'
    'Question: Where is the Eiffel Tower located?\nAnswer:'
)

print('Loading final checkpoint for sanity check ...')
base_check  = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto')
final_check = PeftModel.from_pretrained(base_check, f'{CHECKPOINT_DIR}/final',
                                         is_trainable=False)
final_check.eval()

inputs = tokenizer(TEST_PROMPT, return_tensors='pt').to(final_check.device)
with torch.no_grad():
    out = final_check.generate(**inputs, max_new_tokens=15, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                            skip_special_tokens=True).strip()

print(f'\nPrompt : {TEST_PROMPT}')
print(f'Output : {response}')
print()
print('This is a single-item sanity check, not the trajectory metric.')
print('The actual rise-then-fall pattern is only visible across the full')
print('checkpoint trajectory — that is what Phase 3 measures.')

del base_check, final_check
torch.cuda.empty_cache()


---
## Next Steps

**This notebook does not need to be re-run before Phase 3.** All
checkpoints, training metrics, and the run manifest are saved at:

```
cpi_study/checkpoints/{RUN_TAG}/       ← all checkpoint adapters
cpi_study/results/{RUN_TAG}/           ← train_metrics.json, run_manifest.json
```

Open **`03_Trajectory_Evaluation.ipynb`** next. Set the same `MODEL_NAME`
and `SEED` there (matching `RUN_TAG = "llama31_8b_seed0"` in this example)
so it finds this run's checkpoints on Drive.

**If you plan to run more configurations** (seed 1, or the Mistral
secondary model), come back to this notebook, change `SEED` or
`MODEL_NAME` in Section 2, and run the whole notebook again — it will
start a fresh, independent run under its own `RUN_TAG`.